# HAKE-MER — RoBERTa M2 mechanistic controls

Runs **both** variants on **RoBERTa-base** (3 seeds each):

1. `no_enc` — M1+M2 without inter-emotion Transformer
2. `no_xattn` — M1+M2 without phrase cross-attention

Compare to **M1+M2 reference: 0.526 ± 0.007** (already archived).

After download: integrate zip → `python3 scripts/render_m2_controls_tex.py` → rebuild PDF.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"
BACKBONE = "roberta-base"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
811b977


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.6 MB/s eta 0:00:00


In [4]:
!./run_m1_m2_controls_campaign.sh --variant no_enc --backbone {BACKBONE} {TRAIN_FLAGS}
!./run_m1_m2_controls_campaign.sh --variant no_xattn --backbone {BACKBONE} {TRAIN_FLAGS}

config.json: 100% 481/481 [00:00<00:00, 4.99MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 332kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 22.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 29.6MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 46.4MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 30.4MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   8% 233k/2.77M [00:00<00:03, 751kB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 5.47MB/s,  268kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 5.54MB/s,  273kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 1.25MB/s, 34.4kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 1.27MB/s, 34.8kB/s  ]

simplified/test-00000

In [5]:
import json
from pathlib import Path

slug = "roberta_base"
for variant in ("no_enc", "no_xattn"):
    path = Path(f"reference/artifacts/m1_m2_{variant}_{slug}_campaign.json")
    c = json.loads(path.read_text(encoding="utf-8"))
    m = c["test_aggregate"]["f1_macro"]
    print(path.name, f"F1-macro={m['mean']:.4f}±{m['std']:.4f}")

m1_m2_no_enc_roberta_base_campaign.json F1-macro=0.5067±0.0109
m1_m2_no_xattn_roberta_base_campaign.json F1-macro=0.5236±0.0200


In [6]:
import zipfile
from google.colab import files

slug = "roberta_base"
out_name = "roberta_m2_controls_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for variant in ("no_enc", "no_xattn"):
        campaign = Path(f"reference/artifacts/m1_m2_{variant}_{slug}_campaign.json")
        zf.write(campaign, campaign.name)
        for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_{variant}/metrics.json")):
            zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download roberta_m2_controls_campaign.zip (14.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>